# Real DINOv2 run — MVTec LOCO Anomaly Inspector (Colab GPU)

This notebook re-runs your pipeline with **real `facebook/dinov2-small` patch features** instead of the
handcrafted fallback, on a free GPU, then exports the result CSVs to download back to your PC.

### Before you run — two one-time setup steps
1. **Turn on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator = T4 GPU → Save.**
2. **Make your dataset visible to Colab.** Your `Phase 1` folder lives in Google Drive under
   *Other computers* (backups), which Colab does **not** mount automatically. Easiest fix:
   - Open **drive.google.com** → find the **`Phase 1`** folder (under *Computers → My Computer → … → Data Analaytics Lab*).
   - **Right-click it → Organize → Add shortcut to Drive → My Drive → Add.**
   - It now appears in **My Drive**, which Colab can mount. (Cell 4 will auto-find it.)

Then just **Runtime → Run all**. Total time ≈ **40–80 min** (the nearest-neighbour search on CPU is the slow part).
Keep the tab open so the session doesn't disconnect.


## 1. Check the GPU

In [1]:
import torch
print("torch:", torch.__version__)
assert torch.cuda.is_available(), (
    "No GPU! Set Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU :", torch.cuda.get_device_name(0))


torch: 2.11.0+cu128
GPU : Tesla T4


## 2. Dependencies (transformers / DINOv2)

In [2]:
# Colab ships torch+CUDA. Ensure a transformers with DINOv2 support is importable.
try:
    from transformers import AutoImageProcessor, AutoModel
    import transformers; print("transformers:", transformers.__version__)
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.40", "huggingface_hub>=0.20"], check=True)
    from transformers import AutoImageProcessor, AutoModel
    import transformers; print("installed transformers:", transformers.__version__)


transformers: 5.0.0


## 3. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 4. Locate & verify your dataset
Set `PHASE1_DRIVE` if auto-detection fails. The check below confirms all 5 categories are present.

In [4]:
import os, glob

CATEGORIES = ["breakfast_box", "juice_bottle", "pushpins", "screw_bag", "splicing_connectors"]

def looks_like_phase1(p):
    try:
        return p and all(os.path.isdir(os.path.join(p, c, "test")) for c in CATEGORIES)
    except Exception:
        return False

# 0) set this manually if needed, e.g. "/content/drive/MyDrive/Phase 1"
PHASE1_DRIVE = "/content/drive/Othercomputers/My Computer/8th Trimester 261 Spring 2026/Data Analaytics Lab/Phase 1"

guesses = [PHASE1_DRIVE,
           "/content/drive/MyDrive/Phase 1",
           "/content/drive/MyDrive/Data Analaytics Lab/Phase 1",
           "/content/drive/MyDrive/Data Analytics Lab/Phase 1"]
found = next((g for g in guesses if looks_like_phase1(g)), None)

if not found:  # shallow search of My Drive for a folder named 'Phase 1'
    for hit in glob.glob("/content/drive/MyDrive/**/Phase 1", recursive=True):
        if looks_like_phase1(hit):
            found = hit; break

assert found, ("Could not find 'Phase 1' with the 5 categories. Do the 'Add shortcut to Drive' step "
               "in the intro, or set PHASE1_DRIVE above to the exact path and re-run this cell.")
PHASE1_DRIVE = found
print("Found dataset at:", PHASE1_DRIVE)
for c in CATEGORIES:
    n = len(glob.glob(os.path.join(PHASE1_DRIVE, c, "train", "good", "*.png")))
    print(f"  {c:22s} train/good = {n}")


Found dataset at: /content/drive/Othercomputers/My Computer/8th Trimester 261 Spring 2026/Data Analaytics Lab/Phase 1
  breakfast_box          train/good = 351
  juice_bottle           train/good = 335
  pushpins               train/good = 372
  screw_bag              train/good = 360
  splicing_connectors    train/good = 360


## 5. Copy the data to local Colab disk
Drive is slow for thousands of small reads; copying once (~5.7 GB) makes preprocessing fast and reliable.

In [6]:
import os, subprocess, time
PHASE1_LOCAL = "/content/Phase 1"
if looks_like_phase1(PHASE1_LOCAL):
    print("Local copy already present:", PHASE1_LOCAL)
else:
    t = time.time()
    print("Copying dataset to", PHASE1_LOCAL, "...")
    r = subprocess.run(["cp", "-r", PHASE1_DRIVE, PHASE1_LOCAL])
    if r.returncode != 0 or not looks_like_phase1(PHASE1_LOCAL):
        import shutil
        if os.path.exists(PHASE1_LOCAL): shutil.rmtree(PHASE1_LOCAL, ignore_errors=True)
        shutil.copytree(
    PHASE1_DRIVE,
    PHASE1_LOCAL,
    ignore=shutil.ignore_patterns('*.gdoc', '*.gsheet', '*.gslides', '*.gform', '*.gdraw')
)
    assert looks_like_phase1(PHASE1_LOCAL), "Copy failed."
    print(f"Done in {time.time()-t:.0f}s")


Local copy already present: /content/Phase 1


## 6. Get the pipeline code & build the folder layout
Clones your repo into the `Project_A_LOCO_AD` slot and symlinks `Phase 1` next to it, so
`get_project_paths()` resolves correctly. If your repo is private, the cell tells you how to upload the
one file it needs instead.

In [7]:
import os, sys, subprocess
LAB_ROOT = "/content/lab"
PROJ = os.path.join(LAB_ROOT, "Project_A_LOCO_AD")
os.makedirs(LAB_ROOT, exist_ok=True)

# 6a. get the code (clone repo; fall back to manual upload)
if not os.path.exists(os.path.join(PROJ, "01_notebooks", "loco_project_utils.py")):
    rc = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Az-main/Data-Analytics-Lab-Project.git", PROJ]).returncode
    if rc != 0 or not os.path.exists(os.path.join(PROJ, "01_notebooks", "loco_project_utils.py")):
        os.makedirs(os.path.join(PROJ, "01_notebooks"), exist_ok=True)
        print("Repo clone failed (private?). Upload loco_project_utils.py now:")
        from google.colab import files
        up = files.upload()
        for fn in up:
            os.replace(fn, os.path.join(PROJ, "01_notebooks", "loco_project_utils.py"))
print("engine present:", os.path.exists(os.path.join(PROJ, "01_notebooks", "loco_project_utils.py")))

# 6b. symlink Phase 1 next to the project (use the local copy)
link = os.path.join(LAB_ROOT, "Phase 1")
if not os.path.exists(link):
    os.symlink(PHASE1_LOCAL, link)
print("lab_root contents:", sorted(os.listdir(LAB_ROOT)))

# 6c. import the engine
sys.path.insert(0, os.path.join(PROJ, "01_notebooks"))
import loco_project_utils as L
print("imported loco_project_utils OK")


Repo clone failed (private?). Upload loco_project_utils.py now:


engine present: False
lab_root contents: ['Phase 1', 'Project_A_LOCO_AD']


ModuleNotFoundError: No module named 'loco_project_utils'

## 7. Force REAL DINOv2 features (cosine recipe, no silent fallback)
This replaces `extract_patch_features` with a DINOv2-only extractor that:
- uses `facebook/dinov2-small` (set `MODEL_SIZE='base'` for higher accuracy — ~2× slower NN),
- **L2-normalizes** patch tokens so the pipeline's Euclidean NN becomes **cosine** (AnomalyDINO recipe),
- tags outputs honestly as `dinov2-small`/`dinov2-base`,
- **raises** if the model can't load (so results can never silently fall back to handcrafted).

In [ ]:
import math, numpy as np, torch
from transformers import AutoImageProcessor, AutoModel
from PIL import Image

MODEL_SIZE = "small"            # "small" (fast, default) or "base" (better, ~2x slower NN)
NORMALIZE  = True               # L2-normalize -> Euclidean NN == cosine
MODEL_NAME = f"facebook/dinov2-{MODEL_SIZE}"
MODEL_TAG  = f"dinov2-{MODEL_SIZE}"
_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_orig_simple = L.simple_patch_features

def dinov2_extract(image_path, backend="auto", grid=24, model_cache=None, include_xy=False):
    if backend not in {"auto", "dinov2"}:
        feats, coords, shape = _orig_simple(image_path, grid=grid, include_xy=include_xy)
        return feats, coords, shape, "fallback_patch_stats"
    cache = model_cache if model_cache is not None else {}
    if "model" not in cache:
        cache["processor"] = AutoImageProcessor.from_pretrained(MODEL_NAME)
        cache["model"] = AutoModel.from_pretrained(MODEL_NAME).to(_DEVICE).eval()
    processor, model = cache["processor"], cache["model"]
    with Image.open(image_path) as im:
        inputs = processor(images=im.convert("RGB"), return_tensors="pt").to(_DEVICE)
    with torch.no_grad():
        out = model(**inputs).last_hidden_state[0, 1:, :]   # drop CLS -> [num_patches, dim]
        if NORMALIZE:
            out = torch.nn.functional.normalize(out, dim=1)
        patches = out.float().cpu().numpy()
    g = int(round(math.sqrt(patches.shape[0])))
    coords = np.array([(i, j) for i in range(g) for j in range(g)], dtype=np.int16)
    shape = (g, g)
    if include_xy:
        xy = coords.astype(np.float32).copy()
        xy[:, 0] /= max(1, g - 1); xy[:, 1] /= max(1, g - 1)
        patches = np.concatenate([patches, xy], axis=1)
    return patches.astype(np.float32), coords, shape, MODEL_TAG

L.extract_patch_features = dinov2_extract   # module-global swap; stages pick this up

# smoke test on one image
import glob
_sample = glob.glob(os.path.join(PHASE1_LOCAL, "breakfast_box", "train", "good", "*.png"))[0]
_f, _c, _s, _tag = L.extract_patch_features(_sample, backend="dinov2")
print(f"OK: features {_f.shape} (grid {_s}), backend tag = {_tag}, device = {_DEVICE}")
assert _tag.startswith("dinov2"), "DINOv2 extractor not active!"


## 8. Preprocess (letterbox 384²) — builds the cleaned dataset the methods read

In [ ]:
import time
L.set_reproducible_seed(42)
t = time.time(); print("Preprocessing...")
res = L.run_stage01_preprocessing(lab_root=LAB_ROOT)
print(f"Done in {time.time()-t:.0f}s")


## 9. Global baseline (EfficientAD proxy — image-level, not DINOv2 by design)

In [ ]:
import time
t = time.time(); L.run_stage04_efficientad(lab_root=LAB_ROOT); print(f"stage04 done in {time.time()-t:.0f}s")


## 10–13. The DINOv2 detectors (this is the slow part — NN search on CPU)

In [ ]:
import time
for name, fn in [("stage05 PatchCore(DINOv2)",   lambda: L.run_stage05_patchcore(LAB_ROOT, backend='dinov2')),
                 ("stage06 DINOv2 PatchMemory",  lambda: L.run_stage06_dinov2_patchmemory(LAB_ROOT, backend='dinov2')),
                 ("stage07 GridAware(DINOv2)",    lambda: L.run_stage07_gridaware(LAB_ROOT, backend='dinov2')),
                 ("stage08 CompositionHist(DINOv2)", lambda: L.run_stage08_composition_histogram(LAB_ROOT, backend='dinov2'))]:
    t = time.time(); fn(); print(f"{name} done in {time.time()-t:.0f}s")


## 14. Fusion + final tables

In [ ]:
import time
t = time.time()
L.run_stage09_fusion(lab_root=LAB_ROOT)
L.run_stage10_final_tables(lab_root=LAB_ROOT)
print(f"fusion + tables done in {time.time()-t:.0f}s")


## 15. Verify the run is REAL DINOv2, and preview results

In [ ]:
import pandas as pd
paths = L.get_project_paths(lab_root=LAB_ROOT)
runtime_csvs = {
    "PatchCore":          paths.baseline_root / "PatchCore" / "patchcore_runtime_memory.csv",
    "DINOv2 PatchMemory": paths.method_root / "DINOv2_PatchMemory" / "dinov2_patchmemory_runtime_memory.csv",
    "GridAware":          paths.method_root / "GridAware_DINOv2" / "gridaware_runtime_memory.csv",
    "CompositionHist":    paths.method_root / "CompositionHistogram" / "composition_hist_runtime_memory.csv",
}
print("=== feature_backend check (must be dinov2-*) ===")
ok = True
for name, p in runtime_csvs.items():
    if p.exists():
        b = pd.read_csv(p)["feature_backend"].iloc[0]
        flag = "OK" if str(b).startswith("dinov2") else "  <-- NOT DINOv2!"
        ok &= str(b).startswith("dinov2")
        print(f"  {name:20s}: {b}{'' if flag=='OK' else flag}")
print("ALL DINOv2:", ok)

mr = paths.method_root / "Final_Evaluation" / "main_results_table.csv"
if mr.exists():
    print("\n=== main_results_table.csv ===")
    print(pd.read_csv(mr).to_string(index=False))


## 15b. (Optional) DINOv2 feature-dimension selection — Criterion 4, Part B
Treats the **384 DINOv2 dimensions as features**. Fits PCA on normal patches (feature-importance via explained
variance), then compares a **PCA-reduced** PatchCore (32/64/128 dims) against the **full 384-dim** model.
Outputs `feature_dim_selection_table.csv` + a figure into `Final_Evaluation/`, so they ride back in the results zip.
Takes ~5–15 min on the T4. Skip it if you're short on time — the branch-level feature selection is already done on the PC side.

In [ ]:
# 16b. (Optional) DINOv2 feature-DIMENSION selection study  -- Criterion 4, Part B
# Treats the 384 DINOv2 dims as features: PCA importance + reduced-dim vs full-384 PatchCore.
# Saves feature_dim_selection_table.csv (+ figure) into Final_Evaluation so it ships in the zip.
import os, glob, time, numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

paths     = L.get_project_paths(lab_root=LAB_ROOT)
proj_root = paths.method_root.parent                       # .../Project_A_LOCO_AD
OUT_FS    = paths.method_root / "Final_Evaluation"; OUT_FS.mkdir(parents=True, exist_ok=True)
CATS      = ["breakfast_box","juice_bottle","pushpins","screw_bag","splicing_connectors"]
DIMS      = [32, 64, 128, 384]     # 384 = full (no reduction)
N_MEM     = 80                      # train/good images used for the memory bank (speed cap)
N_VAL     = 40                      # validation/good images for score normalisation
MAX_MEM   = 50000                   # max patches in the memory bank

def _find_clean():
    c = os.path.join(str(proj_root), "03_cleaned_data", "loco_cleaned_letterbox_384")
    if os.path.isdir(c): return c
    h = glob.glob(os.path.join(LAB_ROOT, "**", "loco_cleaned_letterbox_384"), recursive=True)
    return h[0] if h else None
CLEAN = _find_clean()
assert CLEAN, "cleaned dataset not found - run stage01 preprocessing first (cell 8)."

_cache = {}                         # shared so the model loads ONCE
def feats(paths_list):
    out = []
    for p in paths_list:
        f, _, _, _ = L.extract_patch_features(p, backend="dinov2", model_cache=_cache)
        out.append(np.asarray(f, dtype=np.float32))
    return out

def topk_mean(d, frac=0.01):
    d = np.sort(d.ravel()); k = max(1, int(frac*len(d)))
    return float(np.mean(d[-k:]))

def l2n(X):
    n = np.linalg.norm(X, axis=1, keepdims=True); n[n==0]=1.0; return X/n

t0 = time.time(); rows = []; var_rows = []
for cat in CATS:
    g = lambda split, defect: sorted(glob.glob(os.path.join(CLEAN, cat, split, defect, "*.png")))
    mem_imgs = g("train","good")[:N_MEM]
    val_imgs = g("validation","good")[:N_VAL]
    test_specs = ([(p,0) for p in g("test","good")] +
                  [(p,1) for p in g("test","logical_anomalies")] +
                  [(p,1) for p in g("test","structural_anomalies")])
    if not (mem_imgs and val_imgs and test_specs):
        print("  skip", cat, "(missing split)"); continue
    mem_f, val_f = feats(mem_imgs), feats(val_imgs)
    test_f = feats([p for p,_ in test_specs])
    y = np.array([lab for _,lab in test_specs])

    mem_all = np.concatenate(mem_f, 0)
    if len(mem_all) > MAX_MEM:
        idx = np.random.RandomState(42).choice(len(mem_all), MAX_MEM, replace=False); mem_all = mem_all[idx]
    pca_full = PCA(n_components=min(384, mem_all.shape[1])).fit(mem_all)
    evr = np.cumsum(pca_full.explained_variance_ratio_)
    var_rows.append({"category":cat, "dims_for_90pct":int(np.searchsorted(evr,0.90)+1),
                     "dims_for_95pct":int(np.searchsorted(evr,0.95)+1)})

    for k in DIMS:
        if k >= mem_all.shape[1]:
            proj = lambda X: l2n(X)                         # full, no reduction
        else:
            pk = PCA(n_components=k).fit(mem_all)
            proj = lambda X, _p=pk: l2n(_p.transform(X))
        nn = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(proj(mem_all))
        valg = np.array([topk_mean(nn.kneighbors(proj(f))[0]) for f in val_f])
        mu, sd = float(valg.mean()), float(valg.std()+1e-8)
        sc = np.array([(topk_mean(nn.kneighbors(proj(f))[0])-mu)/sd for f in test_f])
        rows.append({"category":cat, "dims":k,
                     "auroc_overall": float(roc_auc_score(y, sc)) if len(set(y))>1 else np.nan})
    print(f"  {cat:20s} done ({time.time()-t0:.0f}s)")

dim_df  = pd.DataFrame(rows)
full    = dim_df[dim_df.dims==384].groupby("dims")["auroc_overall"].mean().iloc[0]
summary = dim_df.groupby("dims")["auroc_overall"].mean().reset_index().rename(columns={"auroc_overall":"mean_overall_auroc"})
summary["pct_of_full_384"] = (summary["mean_overall_auroc"] / full * 100).round(1)
summary.to_csv(OUT_FS / "feature_dim_selection_table.csv", index=False)
dim_df.to_csv(OUT_FS / "feature_dim_selection_per_category.csv", index=False)
pd.DataFrame(var_rows).to_csv(OUT_FS / "feature_dim_pca_variance.csv", index=False)

try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.4,3.4))
    ax.plot(summary["dims"], summary["mean_overall_auroc"], "o-", color="#1AB3A3")
    for _, r in summary.iterrows():
        ax.annotate(f"{r.mean_overall_auroc:.3f}", (r.dims, r.mean_overall_auroc),
                    textcoords="offset points", xytext=(0,8), ha="center", fontsize=9)
    ax.set_xlabel("DINOv2 feature dimensions (PCA)"); ax.set_ylabel("Overall AUROC (5-cat avg)")
    ax.set_title("Feature-dimension selection: PCA-reduced vs full DINOv2 PatchCore")
    fig.tight_layout()
    figdir = os.path.join(str(proj_root), "07_paper_draft", "figures"); os.makedirs(figdir, exist_ok=True)
    fig.savefig(os.path.join(figdir, "figure_feature_dim_selection.png"), dpi=160); plt.close(fig)
except Exception as e:
    print("figure skipped:", e)

print("\n=== DINOv2 feature-dimension selection (PCA-reduced vs full 384) ===")
print(summary.to_string(index=False))
print("\nPCA variance (dims needed for 90%/95%):"); print(pd.DataFrame(var_rows).to_string(index=False))
print("\nSaved to:", OUT_FS / "feature_dim_selection_table.csv")


## 16. Package results → save to Drive + download

In [ ]:
import shutil, os
paths = L.get_project_paths(lab_root=LAB_ROOT)
stage_dir = "/content/loco_dinov2_results"
if os.path.exists(stage_dir): shutil.rmtree(stage_dir)
os.makedirs(stage_dir)
# copy the two result trees (CSVs, configs, maps) that the dashboard + report rebuild from
shutil.copytree(paths.baseline_root, os.path.join(stage_dir, "05_baselines"))
shutil.copytree(paths.method_root,  os.path.join(stage_dir, "06_method_results"))
zip_path = shutil.make_archive("/content/loco_dinov2_results", "zip", stage_dir)
size_mb = os.path.getsize(zip_path) / 1e6
print(f"Created {zip_path} ({size_mb:.1f} MB)")
# save a copy to Drive so it survives a disconnect
try:
    shutil.copy(zip_path, "/content/drive/MyDrive/loco_dinov2_results.zip")
    print("Saved to Drive: MyDrive/loco_dinov2_results.zip")
except Exception as e:
    print("Drive copy skipped:", e)
from google.colab import files
files.download(zip_path)


## ✅ Done — bring the results back
You now have **`loco_dinov2_results.zip`** (downloaded and in your Drive).

Tell Claude on your PC *"the DINOv2 results are downloaded"* and point it to the zip. It will:
1. unzip the new score CSVs into your project (`05_baselines/`, `06_method_results/`),
2. regenerate the dashboard + report + slides with the real DINOv2 numbers,
3. update the method-name labels and remove the "handcrafted" caveat.

**Honest caveats to keep in the report:** features are real DINOv2 (cosine NN), but letterbox padding is not
masked out, and results are single-seed. Both are noted as future work.